# fraQtl Diagnostic — Quickstart

Fingerprint any transformer's compression potential in a few minutes.

**Colab tip:** set runtime to a T4 GPU (Runtime → Change runtime type → T4).
This notebook runs end-to-end on the free T4 in ~5 min for a 0.5–1 B model.

## 1. Install

In [ ]:
# Once fraqtl-diagnostic is on PyPI:
# !pip install -q fraqtl-diagnostic

# While in pre-release (current):
!pip install -q git+https://github.com/fraqtl-ai/fraqtl-diagnostic.git

## 2. Analyze a model

Pick any HuggingFace transformer id. Small-open models (Qwen2.5-0.5B,
TinyLlama-1.1B) run without any HF auth. Gated models (Llama, Mistral) need
you to `huggingface-cli login` first.

In [ ]:
from fraqtl_diagnostic import analyze

MODEL_ID = "Qwen/Qwen2.5-0.5B"   # try also TinyLlama/TinyLlama-1.1B-Chat-v1.0

report = analyze(
    MODEL_ID,
    n_seqs=16,
    seq_len=256,
    projections=("down_proj", "o_proj"),
)

## 3. Summary

In [ ]:
print(report.summary())

## 4. Save and inspect the figure

In [ ]:
report.to_png("fingerprint.png")

from IPython.display import Image
Image("fingerprint.png")

## 5. Save machine-readable reports

In [ ]:
report.to_json("fingerprint.json")
report.to_html("fingerprint.html")

# Colab users: files panel on the left → fingerprint.html / .json / .png

## 6. Walk the per-layer data

Every layer/projection is a `LayerFingerprint` with the full Shannon spectrum.

In [ ]:
import pandas as pd

rows = [f.to_row() for f in report.fingerprints]
df = pd.DataFrame(rows)
df.head(10)

## 7. Interpretation (quick reference)

| Metric | Typical range | Meaning |
|---|---|---|
| γ (down_proj) | 0.7 – 1.0 | stretched-exp shape of MLP Hessian |
| γ (o_proj)    | 0.3 – 0.7 | attention out-projection is usually more stretched |
| k95 / dim     | 10–30 %   | fraction of directions needed for 95% energy |
| headroom      | 0.0 – 1.0 | high = more compressible |

The **suggested bit budget** is a Shannon ceiling. Real compression loss
depends on the recipe (sign correction, rank protection, per-model
calibration) — that's the fraQtl engine, not this tool.

See the README's *How to read the output* section for more.